# Step 1. Download the budgets

The federal budget is published as web pages on budget.canada.ca, one page per chapter. This notebook saves the
chapters of Budgets 2021 to 2026 in `raw/<year>/`, exactly as the site sends them, so every later step reads the same
files and gives the same result when it is run again.

- **Which pages:** the chapters listed in each budget's table of contents (`chap1-en.html`, `chap2-en.html`, ...;
  Budget 2021 groups its chapters into four parts, `p1-en.html` to `p4-en.html`). The introduction, the economic
  overview and the annexes are not measures, so they are left out.
- **Already downloaded pages are kept.** Set `REFRESH = True` to download everything again (the site sometimes corrects
  a page after budget day; `raw/sources.csv` records when each page was saved and a fingerprint of its content).
- The pages are UTF-8 but the server doesn't say so, so the raw bytes are saved, not decoded text.

Output: `raw/<year>/*.html` and `raw/sources.csv`.

In [1]:
import csv
import hashlib
import os
import re
import time
import warnings
from datetime import date

# this Mac's Python 3.9 is built with LibreSSL, so urllib3 warns on import; the downloads are unaffected
warnings.filterwarnings("ignore", message="urllib3 v2 only supports OpenSSL")
import requests  # noqa: E402 (after the warning filter)
from bs4 import BeautifulSoup

YEARS = [2021, 2022, 2023, 2024, 2025, 2026]
BASE = "https://budget.canada.ca/{year}/report-rapport/{page}"
RAW = "raw"
REFRESH = False                                   # True: download every page again
HEADERS = {"User-Agent": "Mozilla/5.0 (budget galaxy; research)"}

## Download one page
A page is fetched only if it isn't in `raw/` yet (or `REFRESH` is on). There is a short pause between requests, to be
polite to the site.

In [2]:
def fetch(year, page):
    """Save one page of a budget in raw/<year>/ and return its path."""
    path = os.path.join(RAW, str(year), page)
    if os.path.exists(path) and not REFRESH:
        return path                               # already saved: keep it, so re-runs read the same file
    os.makedirs(os.path.dirname(path), exist_ok=True)
    r = requests.get(BASE.format(year=year, page=page), headers=HEADERS, timeout=60)
    r.raise_for_status()                          # stop on a missing page rather than carry on without it
    with open(path, "wb") as f:
        f.write(r.content)                        # raw bytes (UTF-8 without a charset header)
    time.sleep(0.5)
    return path

## Each budget: its table of contents, then its chapters

In [3]:
rows = []
for year in YEARS:
    toc = fetch(year, "toc-tdm-en.html")                                  # the table of contents
    soup = BeautifulSoup(open(toc, "rb").read(), "html.parser")
    pages = set()
    for a in soup.find_all("a", href=True):
        m = re.match(r"^((?:chap|p)\d+)-en\.html", a["href"])           # chapter (or part) pages only
        if m:
            pages.add(m.group(1) + "-en.html")
    pages = sorted(pages, key=lambda p: int(re.search(r"\d+", p).group()))
    for page in ["toc-tdm-en.html"] + pages:
        path = fetch(year, page)
        data = open(path, "rb").read()
        saved = date.fromtimestamp(os.path.getmtime(path)).isoformat()
        rows.append({"year": year, "page": page, "url": BASE.format(year=year, page=page), "bytes": len(data),
                     "sha256": hashlib.sha256(data).hexdigest()[:16], "saved": saved})
    print(f"Budget {year}: {len(pages)} chapter pages ({', '.join(pages)})")

Budget 2021: 4 chapter pages (p1-en.html, p2-en.html, p3-en.html, p4-en.html)
Budget 2022: 9 chapter pages (chap1-en.html, chap2-en.html, chap3-en.html, chap4-en.html, chap5-en.html, chap6-en.html, chap7-en.html, chap8-en.html, chap9-en.html)
Budget 2023: 6 chapter pages (chap1-en.html, chap2-en.html, chap3-en.html, chap4-en.html, chap5-en.html, chap6-en.html)
Budget 2024: 8 chapter pages (chap1-en.html, chap2-en.html, chap3-en.html, chap4-en.html, chap5-en.html, chap6-en.html, chap7-en.html, chap8-en.html)
Budget 2025: 5 chapter pages (chap1-en.html, chap2-en.html, chap3-en.html, chap4-en.html, chap5-en.html)
Budget 2026: 6 chapter pages (chap1-en.html, chap2-en.html, chap3-en.html, chap4-en.html, chap5-en.html, chap6-en.html)


## The list of pages, with a fingerprint of each
If a page changes on the site and is downloaded again, its fingerprint (`sha256`) changes too.

In [4]:
with open(os.path.join(RAW, "sources.csv"), "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0]))
    w.writeheader()
    w.writerows(rows)
print(f"{len(rows)} pages listed in raw/sources.csv, {sum(r['bytes'] for r in rows) / 1e6:.1f} MB in all")

44 pages listed in raw/sources.csv, 5.6 MB in all
